In [1]:
%pip install -U google-generativeai
%pip install  google-ai-generativelanguage==0.6.15
%pip install -U langchain-google-genai 
%pip install -U langchain-community
%pip install -U langgraph




[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
  Using cached filetype-1.2.0-py2.py3-none-any.whl.metadata (6.5 kB)
  Using cached jsonpatch-1.33-py2.py3-none-any.whl.metadata (3.0 kB)
  Using cached pyyaml-6.0.3-cp314-cp314-macosx_11_0_arm64.whl.metadata (2.4 kB)
  Using cached uuid_utils-0.14.1-cp39-abi3-macosx_10_12_x86_64.macosx_11_0_arm64.macosx_10_12_universal2.whl.metadata (4.8 kB)
  Using cached orjson-3.11.7-cp314-cp314-macosx_10_15_x86_64.macosx_11_0_arm64.macosx_10_15_universal2.whl.metadata (41 kB)
  Using cached requests_toolbelt-1.0.0-py2.py3-none-any.whl.metadata (14 kB)
  Using cached xxhash-3.6.0-cp314-cp314-macosx_11_0_arm64.whl.metadata (13 kB)
  Using cached zsta

In [2]:
import os 
import re 
import google.generativeai as genai
from langgraph.graph import StateGraph, END
from typing import TypedDict


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/var/folders/l2/t02s0r_51x727q8t6t5_qbp40000gn/T/ipykernel_58983/3776146230.py:3: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [3]:
import os
from dotenv import load_dotenv
import google.generativeai as genai

load_dotenv()

os.environ["GOOGLE_API_KEY"] = os.getenv("GEMINI_API_KEY")
genai.configure(api_key=os.getenv("GEMINI_API_KEY"))
model = genai.GenerativeModel("gemini-2.5-flash")
response = model.generate_content("Hello World")

print(response.text)

Hello there!


In [4]:
client = genai.GenerativeModel("gemini-2.5-flash")

class Agent: 
    def __init__(self, system=""):
        self.system = system
        self.messages = []
        if self.system:
            self.messages.append({"role": "system", "content": self.system})
    
    def __call__(self, message):
        self.messages.append({"role": "user", "content": message})
        result = self.execute()
        self.messages.append({"role": "model", "content": result})
        return result
    
    def execute(self):
        prompt = ""
        for msg in self.messages:
            prompt += f"{msg['role']}: {msg['content']}\n"
        response = client.generate_content(prompt)
        return response.text
    
if __name__ == "__main__":
        agent = Agent(system="Você é um assistente inteligente que responde perguntas de forma clara e concisa.")
        print(agent("Qual a capital da França?"))

A capital da França é Paris.


In [5]:
PROMPT_REACT = """
Você funciona em um ciclo de Pensamento, Ação, Pausa e Observação.
Ao final do ciclo, você fornece uma Resposta.
Use "Pensamento" para descrever seu raciocínio.
Use "Ação" para executar ferramentas - e então retorne "PAUSA".
A "Observação" será o resultado da ação executada.
Ações disponíveis:
  - consultar_estoque: retorna a quantidade disponível de um item no inventário (ex: "consultar_estoque: teclado")
  - consultar_preco_produto: retorna o preço unitário de um produto (ex: "consultar_preco_produto: mouse gamer")

Exemplo:
Pergunta: Quantos monitores temos em estoque?
Pensamento: Devo consultar a ação consultar_estoque para saber a quantidade de monitores.
Ação: consultar_estoque: monitor
PAUSA

Observação: Temos 75 monitores em estoque.
Resposta: Há 75 monitores em estoque.
""".strip()

In [6]:
class AgentState(TypedDict):
    pergunta: str
    historico: list[str]
    acao_pendente: str
    resposta_final: str

In [7]:
def consultar_estoque(item: str) -> str:
    """"
    Simula a consulta de estoque para um item específico no inventário.
    """
    item = item.lower()
    estoque = {
        "monitor": 150,
        "mouse gamer": 80,
        "teclado": 75,
        "webcam": 50,
        "impressora": 30
    }
    if item in estoque:
        return f"Temos {estoque[item]} {item}s em estoque."
    else:
        return f"Desculpe, não temos informações de estoque para {item}."
    
def consultar_preco_produto(produto: str) -> str:
    """"
    Simula a consulta de preço para um produto específico.
    """
    produto = produto.lower()
    precos = {
        "monitor": 1200.00,
        "mouse gamer": 250.00,
        "teclado": 150.00,
        "webcam": 300.00,
        "impressora": 800.00
    }
    if produto in precos:
        return f"O preço do {produto} é R${precos[produto]:.2f}."
    else:
        return f"Desculpe, não temos informações de preço para {produto}."

In [13]:
print(consultar_estoque("monitor"))

Temos 150 monitors em estoque.


In [14]:
print(consultar_preco_produto("monitor"))

O preço do monitor é R$1200.00.


In [8]:
def run_react_agent(pergunta: str, max_interactions: int = 5) -> str:
    model = genai.GenerativeModel("gemini-2.5-flash")
    chat = model.start_chat(history=[])

    chat.send_message(PROMPT_REACT)
    current_prompt = pergunta
    
    for i in range(max_interactions):
        response = chat.send_message(current_prompt)
        response_text = response.text.strip()
        
        print(f"--- Iteração {i+1} ---")
        print (f"Modelo pensou/respondeu:\n{response_text}\n")

        if response_text.startswith("Resposta:"):
            return response_text.replace("Resposta: ", "").strip()
        
        match = re.search(r"Ação:\s*(\w)+:\s*(.*)", response_text)

        if match:
            action_name = match.group(1).strip()
            action_arg = match.group(2).strip()

            observacao = ""
            if action_name == "consultar_estoque":
                observacao = consultar_estoque(action_arg)
            elif action_name == "consultar_preco_produto":
                observacao = consultar_preco_produto(action_arg)
            else:
                observacao = f"Ação desconhecida: {action_name}"    
            current_prompt = f"Observação: {observacao}\nResposta:"

            print(f"Executou ação: {action_name} ({action_arg})")
            print(f"Observação resultante: {observacao}\n")
        else:
            return f"Erro: O agente não conseguiu extrair uma Açãou ou Resposta final após {i+1} interações. Última resposta do modelo:\n{response_text}"
    return f"Erro: O agente atingiu o número máximo de interações ({max_interactions}) sem chegar a uma Resposta final. Última resposta do modelo:\n{response_text}"


In [9]:
pergunta_1 = "Quantidade de impressora no inventário?"
print(f"**Interação 1: {pergunta_1}**")
resposta_1 = run_react_agent(pergunta_1)
print(f"\n**RESPOSTA FINAL DO AGENTE 1:** {resposta_1}\n")

print("\n" + "="*50 + "\n") 

**Interação 1: Quantidade de impressora no inventário?**
--- Iteração 1 ---
Modelo pensou/respondeu:
Pensamento: Devo consultar a ação consultar_estoque para saber a quantidade de impressoras.
Ação: consultar_estoque: impressora
PAUSA

Executou ação: e (impressora)
Observação resultante: Ação desconhecida: e

--- Iteração 2 ---
Modelo pensou/respondeu:
Pensamento: Recebi uma observação de "Ação desconhecida: e". Isso indica que a ação que tentei executar, ou o resultado que deveria vir dela, não foi válido. Preciso retomar o ciclo de pensamento considerando que a ação anterior falhou ou que a observação fornecida não é o resultado esperado de `consultar_estoque: impressora`.

A observação "Ação desconhecida: e" não é o formato esperado para um resultado de estoque. O formato esperado seria algo como "Temos X itens em estoque" ou "Erro ao consultar estoque".

Dada a observação, parece que houve um problema na execução da ferramenta `consultar_estoque` ou na forma como a observação foi f

In [10]:
pergunta_2 = "Qual é o valor da impressora no inventário?"
print(f"**Interação 2: {pergunta_2}**")
resposta_2 = run_react_agent(pergunta_2)
print(f"\n**RESPOSTA FINAL DO AGENTE 2:** {resposta_2}\n")

print("\n" + "="*50 + "\n") 

**Interação 2: Qual é o valor da impressora no inventário?**
--- Iteração 1 ---
Modelo pensou/respondeu:
Pensamento: O usuário deseja saber o valor da impressora. Para isso, devo consultar o preço do produto utilizando a ação `consultar_preco_produto`.
Ação: consultar_preco_produto: impressora
PAUSA

Executou ação: o (impressora)
Observação resultante: Ação desconhecida: o

--- Iteração 2 ---
Modelo pensou/respondeu:
Não foi possível consultar o valor da impressora, pois a ação executada retornou um erro de "Ação desconhecida: o". Por favor, verifique se a ação `consultar_preco_produto` está disponível ou se houve algum problema na execução.


**RESPOSTA FINAL DO AGENTE 2:** Erro: O agente não conseguiu extrair uma Açãou ou Resposta final após 2 interações. Última resposta do modelo:
Não foi possível consultar o valor da impressora, pois a ação executada retornou um erro de "Ação desconhecida: o". Por favor, verifique se a ação `consultar_preco_produto` está disponível ou se houve algu

In [11]:
def consultar_estoque(item: str) -> str: 
    """
    Simula a consulta de estoque de um item no inventário.
    """
    item = item.lower().strip()
    estoque = {
        "monitor": 75,
        "teclado": 120,
        "mouse gamer": 80,
        "webcam": 40,
        "headset": 60,
        "impressora": 15
    }

    if item in estoque:
        return f"Temos {estoque[item]} {item}s em estoque."
    else:
        return f"Item '{item}' não encontrado no inventário."

def consultar_preco_produto(produto: str) -> str: 
    """
    Simula a consulta do preço unitário de um produto.
    """
    produto = produto.lower().strip()
    precos = {
        "monitor": 999.90,
        "teclado": 150.00,
        "mouse gamer": 99.50,
        "webcam": 120.00,
        "headset": 180.00,
        "impressora": 750.00
    }

    if produto in precos:
        return f"O preço de um(a) {produto} é R$ {precos[produto]:.2f}."
    else:
        return f"Produto '{produto}' não encontrado na lista de preços."

def ferramenta_encontrar_produto_mais_caro() -> str:
    """
    Retorna o nome e o preço do produto mais caro no inventário.
    Esta função não precisa de argumentos.
    """
    
    precos_do_inventario = {
        "monitor": 999.90,
        "teclado": 150.00,
        "mouse gamer": 99.50,
        "webcam": 120.00,
        "headset": 180.00,
        "impressora": 750.00
    }

    if not precos_do_inventario: 
        return "Nenhum produto encontrado na lista de preços para comparação."

    
    nome_produto_mais_caro = max(precos_do_inventario, key=precos_do_inventario.get)
    valor_produto_mais_caro = precos_do_inventario[nome_produto_mais_caro]

    return f"O produto mais caro é o(a) {nome_produto_mais_caro} com preço de R$ {valor_produto_mais_caro:.2f}."
    


In [12]:
PROMPT_REACT = """
Você funciona em um ciclo de Pensamento, Ação, Pausa e Observação.
Ao final do ciclo, você fornece uma Resposta.
Use "Pensamento" para descrever seu raciocínio.
Use "Ação" para executar ferramentas - e então retorne "PAUSA".
A "Observação" será o resultado da ação executada.
Ações disponíveis:
  - consultar_estoque: retorna a quantidade disponível de um item no inventário (ex: "consultar_estoque: teclado")
  - consultar_preco_produto: retorna o preço unitário de um produto (ex: "consultar_preco_produto: mouse gamer")
  - encontrar_produto_mais_caro: retorna o nome e o preço do produto mais caro no inventário (não requer argumentos)

Exemplo:
Pergunta: Quantos monitores temos em estoque?
Pensamento: Devo consultar a ação consultar_estoque para saber a quantidade de monitores.
Ação: consultar_estoque: monitor
PAUSA

Observação: Temos 75 monitores em estoque.
Resposta: Há 75 monitores em estoque.

Exemplo:
Pergunta: Qual é o produto mais caro?
Pensamento: Preciso usar a ação encontrar_produto_mais_caro para descobrir qual produto tem o maior preço.
Ação: encontrar_produto_mais_caro
PAUSA

Observação: O produto mais caro é o(a) monitor com preço de R$ 999.90.
Resposta: O produto mais caro é o(a) monitor com preço de R$ 999.90.
""".strip()

In [13]:
def run_react_agent(pergunta: str, max_iterations: int = 5) -> str:
    """
    Executa o ciclo ReAct para uma dada pergunta usando o modelo Gemini.
    """

    model = genai.GenerativeModel('gemini-2.5-flash')

    chat = model.start_chat(history=[])
    chat.send_message(PROMPT_REACT)

    current_prompt = pergunta

    for i in range(max_iterations):
        response = chat.send_message(current_prompt)
        response_text = response.text.strip()

        print(f"\n--- Iteração {i+1} ---")
        print(f"Modelo pensou/respondeu:\n{response_text}\n")

        
        response_match_final = re.search(r"Resposta:\s*(.*)", response_text, re.DOTALL)
        if response_match_final:
            return response_match_final.group(1).strip()

        
        match = re.search(r"Ação:\s*(\w+)(?::\s*([^\n]*))?", response_text)

        if match:
            action_name = match.group(1).strip()

            action_arg = match.group(2).strip() if match.group(2) is not None else ""

            observacao_da_acao = ""

            if action_name == "consultar_estoque":
                observacao_da_acao = consultar_estoque(action_arg)
            elif action_name == "consultar_preco_produto":
                observacao_da_acao = consultar_preco_produto(action_arg)
            elif action_name == "encontrar_produto_mais_caro": 
                observacao_da_acao = ferramenta_encontrar_produto_mais_caro()
            else:
                observacao_da_acao = f"Erro: Ação '{action_name}' desconhecida. Verifique o prompt ou a implementação da ferramenta."

            # Simplificamos o prompt de Observação
            current_prompt = f"Observação: {observacao_da_acao}"
            print(f"Executou ação: {action_name} com argumento '{action_arg}'")
            print(f"Observação: {observacao_da_acao}\n")

        else:
            
            return f"Erro: O agente não conseguiu extrair uma Ação ou Resposta final após {i+1} iterações. Última resposta do modelo: {response_text}"

    
    return "Erro: Limite máximo de iterações atingido sem uma resposta final do agente."

In [14]:
pergunta_1 = "Quantos teclados temos em estoque?"
print(f"\n**Interação 1: {pergunta_1}**")
resposta_1 = run_react_agent(pergunta_1)
print(f"\n**RESPOSTA FINAL DO AGENTE 1:** {resposta_1}\n")



**Interação 1: Quantos teclados temos em estoque?**

--- Iteração 1 ---
Modelo pensou/respondeu:
Pensamento: Devo consultar a ação consultar_estoque para saber a quantidade de teclados.
Ação: consultar_estoque: teclado
PAUSA

Executou ação: consultar_estoque com argumento 'teclado'
Observação: Temos 120 teclados em estoque.


--- Iteração 2 ---
Modelo pensou/respondeu:
Resposta: Há 120 teclados em estoque.


**RESPOSTA FINAL DO AGENTE 1:** Há 120 teclados em estoque.



In [15]:
pergunta_4 = "Qual é o produto mais caro?"
print(f"\n**Interação 4: {pergunta_4}**")
resposta_4 = run_react_agent(pergunta_4)
print(f"\n**RESPOSTA FINAL DO AGENTE 4:** {resposta_4}\n")



**Interação 4: Qual é o produto mais caro?**

--- Iteração 1 ---
Modelo pensou/respondeu:
Pensamento: Preciso usar a ação encontrar_produto_mais_caro para descobrir qual produto tem o maior preço.
Ação: encontrar_produto_mais_caro
PAUSA

Executou ação: encontrar_produto_mais_caro com argumento ''
Observação: O produto mais caro é o(a) monitor com preço de R$ 999.90.


--- Iteração 2 ---
Modelo pensou/respondeu:
Resposta: O produto mais caro é o(a) monitor com preço de R$ 999.90.


**RESPOSTA FINAL DO AGENTE 4:** O produto mais caro é o(a) monitor com preço de R$ 999.90.



In [16]:
def ferramenta_calcular_valor_total_lista(lista_itens: str) -> str: 
    """
    Calcula o valor total de uma lista de itens de compra.
    Recebe uma string com itens separados por vírgula (ex: "monitor, teclado, mouse gamer") e retorna o valor total somando os preços unitários de cada item.
    """
    precos_do_inventario = {
        "monitor": 999.90,
        "teclado": 150.00,
        "mouse gamer": 99.50,
        "webcam": 120.00,
        "headset": 180.00,
        "impressora": 750.00
    }
    itens_processados = [item.strip().lower() for item in lista_itens.split(",")]
    valor_total = 0.0
    itens_desconecidos = [] 

    for item in itens_processados:
        if item in precos_do_inventario:
            valor_total += precos_do_inventario[item]
        else:
            itens_desconecidos.append(item)

    resposta = f"O valor total dos itens encontrados é R$ {valor_total:.2f}."
    if itens_desconecidos:
        resposta += f" Os seguintes itens não foram encontrados no inventário: {', '.join(itens_desconecidos)}."
    return resposta

In [17]:

print("Testando ferramenta_calcular_valor_total_lista:")

lista_1 = "teclado, mouse gamer, monitor"
resultado_1 = ferramenta_calcular_valor_total_lista(lista_1)
print(f"Lista: '{lista_1}'\nResultado: {resultado_1}\n")

Testando ferramenta_calcular_valor_total_lista:
Lista: 'teclado, mouse gamer, monitor'
Resultado: O valor total dos itens encontrados é R$ 1249.40.



In [18]:
def run_react_agent(pergunta: str, max_iterations: int = 5) -> str:
    model = genai.GenerativeModel('gemini-2.5-flash')

    chat = model.start_chat(history=[])
    chat.send_message(PROMPT_REACT)

    current_prompt = pergunta

    for i in range(max_iterations):
        response = chat.send_message(current_prompt)
        response_text = response.text.strip()

        print(f"\n--- Iteração {i+1} ---")
        print(f"Modelo pensou/respondeu:\n{response_text}\n")

        response_match_final = re.search(r"Resposta:\s*(.*)", response_text, re.DOTALL)
        if response_match_final:
            return response_match_final.group(1).strip()


        match = re.search(r"Ação:\s*(\w+)(?::\s*([^\n]*))?", response_text)

        if match:
            action_name = match.group(1).strip()
            action_arg = match.group(2).strip() if match.group(2) is not None else ""

            observacao_da_acao = ""

            if action_name == "consultar_estoque":
                observacao_da_acao = consultar_estoque(action_arg)
            elif action_name == "consultar_preco_produto":
                observacao_da_acao = consultar_preco_produto(action_arg)
            elif action_name == "encontrar_produto_mais_caro":
                observacao_da_acao = ferramenta_encontrar_produto_mais_caro()
            elif action_name == "calcular_valor_total_lista":
                observacao_da_acao = ferramenta_calcular_valor_total_lista(action_arg)
            else:
                observacao_da_acao = f"Erro: Ação '{action_name}' desconhecida. Verifique o prompt ou a implementação da ferramenta."

            current_prompt = f"Observação: {observacao_da_acao}"

            print(f"Executou ação: {action_name} com argumento '{action_arg}'")
            print(f"Observação: {observacao_da_acao}\n")

        else:
            return f"Erro: O agente não conseguiu extrair uma Ação ou Resposta final após {i+1} iterações. Última resposta do modelo: {response_text}"

    return "Erro: Limite máximo de iterações atingido sem uma resposta final do agente."

In [20]:
PROMPT_REACT = """
Você funciona em um ciclo de Pensamento, Ação, Pausa e Observação.
Ao final do ciclo, você fornece uma Resposta.
Use "Pensamento" para descrever seu raciocínio.
Use "Ação" para executar ferramentas - e então retorne "PAUSA".
A "Observação" será o resultado da ação executada.
Ações disponíveis:
  - consultar_estoque: retorna a quantidade disponível de um item no inventário (ex: "consultar_estoque: teclado")
  - consultar_preco_produto: retorna o preço unitário de um produto (ex: "consultar_preco_produto: mouse gamer")
  - encontrar_produto_mais_caro: retorna o nome e o preço do produto mais caro no inventário (não requer argumentos)
  - calcular_valor_total_lista: calcula o valor total de uma lista de itens de compra. Recebe uma string com itens separados por vírgula (ex: "teclado, mouse gamer, monitor")

Exemplo:
Pergunta: Quantos monitores temos em estoque?
Pensamento: Devo consultar a ação consultar_estoque para saber a quantidade de monitores.
Ação: consultar_estoque: monitor
PAUSA

Observação: Temos 75 monitores em estoque.
Resposta: Há 75 monitores em estoque.

Exemplo:
Pergunta: Qual é o produto mais caro?
Pensamento: Preciso usar a ação encontrar_produto_mais_caro para descobrir qual produto tem o maior preço.
Ação: encontrar_produto_mais_caro
PAUSA

Observação: O produto mais caro é o(a) monitor com preço de R$ 999.90.
Resposta: O produto mais caro é o(a) monitor com preço de R$ 999.90.

Exemplo:
Pergunta: Quanto custa um teclado e um mouse gamer?
Pensamento: O usuário quer saber o valor total de vários itens. Devo usar a ação calcular_valor_total_lista com os itens "teclado, mouse gamer".
Ação: calcular_valor_total_lista: teclado, mouse gamer
PAUSA

Observação: O valor total dos itens encontrados é R$ 249.50.
Resposta: O valor total do teclado e do mouse gamer é R$ 249.50.
""".strip()


In [ ]:
print("--- Começando as Interações com o Agente ReAct ---")

# Interação 1: Consultar Estoque
pergunta_1 = "Quantos teclados temos em estoque?"
print(f"\n**Interação 1: {pergunta_1}**")
resposta_1 = run_react_agent(pergunta_1)
print(f"\n**RESPOSTA FINAL DO AGENTE 1:** {resposta_1}\n")

print("\n" + "="*80 + "\n")

# Interação 2: Consultar Preço
pergunta_2 = "Qual o preço de um headset?"
print(f"\n**Interação 2: {pergunta_2}**")
resposta_2 = run_react_agent(pergunta_2)
print(f"\n**RESPOSTA FINAL DO AGENTE 2:** {resposta_2}\n")

print("\n" + "="*80 + "\n")

# Interação 3: Item não encontrado no estoque
pergunta_3 = "Temos cadeiras em estoque?"
print(f"\n**Interação 3: {pergunta_3}**")
resposta_3 = run_react_agent(pergunta_3)
print(f"\n**RESPOSTA FINAL DO AGENTE 3:** {resposta_3}\n")

print("\n" + "="*80 + "\n")

# Interação 4: Encontrar o Produto Mais Caro
pergunta_4 = "Qual é o produto mais caro?"
print(f"\n**Interação 4: {pergunta_4}**")
resposta_4 = run_react_agent(pergunta_4)
print(f"\n**RESPOSTA FINAL DO AGENTE 4:** {resposta_4}\n")

print("\n" + "="*80 + "\n")

# Interação 5: Calcular Valor Total da Lista (NOVA FUNCIONALIDADE)
pergunta_5 = "Qual o valor de um teclado, uma impressora e uma webcam?"
print(f"\n**Interação 5: {pergunta_5}**")
resposta_5 = run_react_agent(pergunta_5)
print(f"\n**RESPOSTA FINAL DO AGENTE 5:** {resposta_5}\n")


print("\n--- Fim das Interações ---")

In [ ]:
def iniciar_conversacao_com_agente():
    print("--- Agente de Inventário Interativo ---")
    print("Digite sua pergunta sobre o inventário, ou digite 'sair' para encerrar.")
    print("-" * 50)

    while True:
        pergunta_usuario = input("\nVocê: ")

        if pergunta_usuario.lower().strip() == 'sair':
            print("Encerrando a conversa. Até logo!")
            break

        print("\nAgente: Processando...")
        try:

            resposta_agente = run_react_agent(pergunta_usuario)
            print(f"\nAgente: {resposta_agente}")
        except Exception as e:

            print(f"\nAgente: Ocorreu um erro ao processar sua pergunta: {e}")
            print("Por favor, tente novamente ou digite 'sair'.")

if __name__ == "__main__":
    iniciar_conversacao_com_agente()